In [2]:
# pii_detector.py मध्ये main() मध्ये add कर

def scan_excel_file(filepath):
    """Scan Excel files for PII"""
    import openpyxl
    wb = openpyxl.load_workbook(filepath)
    all_findings = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        data = []
        for row in ws.iter_rows(values_only=True):
            data.append(list(row))

        df = pd.DataFrame(data[1:], columns=data[0])
        df['source_sheet'] = sheet_name
        findings = scan_dataframe(df)
        all_findings.append(findings)

    return pd.concat(all_findings) if all_findings else pd.DataFrame()

In [3]:
import re
import pandas as pd

def scan_dataframe(df):
    """Scans a Pandas DataFrame for PII patterns."""
    findings = []
    for col in df.select_dtypes(include=['object', 'string']).columns:
        for index, value in df[col].items():
            if pd.notna(value):
                for pii_name, pii_info in PII_PATTERNS.items():
                    matches = re.findall(pii_info['pattern'], str(value))
                    for match in matches:
                        findings.append({
                            'column': col,
                            'row_index': index,
                            'detected_value': match,
                            'pii_type': pii_name,
                            'severity': pii_info['severity'],
                            'gdpr_category': pii_info['gdpr_category'],
                            'source_sheet': df.get('source_sheet', None).iloc[index] if 'source_sheet' in df.columns else None
                        })
    return pd.DataFrame(findings)

In [4]:
import pandas as pd

# Create a sample DataFrame with mock PII data
sample_data = {
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Details': [
        'Passport: A1234567, GSTIN: 07ABCDE1234F1Z5',
        'Bank Account: 123456789012345678, IFSC: ICIC0000001',
        'Just some text without PII'
    ],
    'CustomerID': ['CUST001', 'CUST002', 'CUST003']
}
sample_df = pd.DataFrame(sample_data)

print("Sample DataFrame:")
display(sample_df)

Sample DataFrame:


,Name,Details,CustomerID
0,Alice,"Passport: A1234567, GSTIN: 07ABCDE1234F1Z5",CUST001
1,Bob,"Bank Account: 123456789012345678, IFSC: ICIC00...",CUST002
2,Charlie,Just some text without PII,CUST003


In [5]:
import re
import pandas as pd

# Redefining PII_PATTERNS and scan_dataframe here to ensure they are available for this cell's execution.
# Ideally, cell 9edYtQwkLJcE and e6bbb510 should be run first to define them globally.
PII_PATTERNS = {
"passport_india": {
    "pattern": r'\b[A-PR-WY][1-9]\d\s?\d{4}[1-9]\b',
    "severity": "CRITICAL",
    "gdpr_category": "Government ID"
},
"gstin": {
    "pattern": r'\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b',
    "severity": "HIGH",
    "gdpr_category": "Business ID"
},
"ifsc_code": {
    "pattern": r'\b[A-Z]{4}0[A-Z0-9]{6}\b',
    "severity": "HIGH",
    "gdpr_category": "Financial Data"
},
"bank_account": {
    "pattern": r'\b\d{9,18}\b',
    "severity": "CRITICAL",
    "gdpr_category": "Financial Data"
}
}

def scan_dataframe(df):
    """Scans a Pandas DataFrame for PII patterns."""
    findings = []
    for col in df.select_dtypes(include=['object', 'string']).columns:
        for index, value in df[col].items():
            if pd.notna(value):
                for pii_name, pii_info in PII_PATTERNS.items():
                    matches = re.findall(pii_info['pattern'], str(value))
                    for match in matches:
                        findings.append({
                            'column': col,
                            'row_index': index,
                            'detected_value': match,
                            'pii_type': pii_name,
                            'severity': pii_info['severity'],
                            'gdpr_category': pii_info['gdpr_category'],
                            'source_sheet': df.get('source_sheet', None).iloc[index] if 'source_sheet' in df.columns else None
                        })
    return pd.DataFrame(findings)

# Create a sample DataFrame with mock PII data
sample_data = {
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Details': [
        'Passport: A1234567, GSTIN: 07ABCDE1234F1Z5',
        'Bank Account: 123456789012345678, IFSC: ICIC0000001',
        'Just some text without PII'
    ],
    'CustomerID': ['CUST001', 'CUST002', 'CUST003']
}
sample_df = pd.DataFrame(sample_data)

# Run the scan_dataframe function on the sample DataFrame
pipeline_findings = scan_dataframe(sample_df)

print("\nPII Findings:")
display(pipeline_findings)


PII Findings:


,column,row_index,detected_value,pii_type,severity,gdpr_category,source_sheet
0,Details,0,A1234567,passport_india,CRITICAL,Government ID,None
1,Details,0,07ABCDE1234F1Z5,gstin,HIGH,Business ID,None
2,Details,1,ICIC0000001,ifsc_code,HIGH,Financial Data,None
3,Details,1,123456789012345678,bank_account,CRITICAL,Financial Data,None


In [6]:
# pii_detector.py मध्ये PII_PATTERNS मध्ये add कर

PII_PATTERNS = {
"passport_india": {
    "pattern": r'\b[A-PR-WY][1-9]\d\s?\d{4}[1-9]\b',
    "severity": "CRITICAL",
    "gdpr_category": "Government ID"
},
"gstin": {
    "pattern": r'\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b',
    "severity": "HIGH",
    "gdpr_category": "Business ID"
},
"ifsc_code": {
    "pattern": r'\b[A-Z]{4}0[A-Z0-9]{6}\b',
    "severity": "HIGH",
    "gdpr_category": "Financial Data"
},
"bank_account": {
    "pattern": r'\b\d{9,18}\b',
    "severity": "CRITICAL",
    "gdpr_category": "Financial Data"
}
}

In [ ]:
# config.py नवीन file बनव
CONFIG = {
    "contamination": 0.05,
    "n_estimators": 200,
    "random_state": 42,
    "severity_thresholds": {
        "critical": 0.6,
        "high": 0.45,
        "medium": 0.0
    },
    "features": [
        "login_attempts",
        "unique_ips",
        "off_hours",
        "failed_auths",
        "attempt_ip_ratio",
        "failure_rate",
        "risk_score",
        "data_accessed_mb"
    ]
}